In [2]:
# Factory Reallocation & Shipping Optimization Recommendation System
# Data Cleaning & Preprocessing

In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [4]:
df = pd.read_csv("../data/raw/Nassau Candy Distributor.csv")

load mapping files

In [5]:
factory_map = pd.read_csv(
    "../data/external/product_factory_mapping.csv"
)

factory_coords = pd.read_csv(
    "../data/external/factory_coordinates.csv"
)

In [6]:
# dataset copy for cleaning

In [7]:
df_clean = df.copy()

# columns name check

In [8]:
df_clean.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Country/Region', 'City', 'State/Province',
       'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name',
       'Sales', 'Units', 'Gross Profit', 'Cost'],
      dtype='str')

# Convert Date Columns

In [9]:
df_clean[['Order Date','Ship Date']].head()

,Order Date,Ship Date
0,03-01-2024,30-06-2026
1,04-01-2024,01-07-2026
2,04-01-2024,01-07-2026
3,04-01-2024,01-07-2026
4,05-01-2024,05-07-2026


In [11]:
df_clean['Order Date'] = pd.to_datetime(
    df_clean['Order Date'],
    format='%d-%m-%Y'
)

df_clean['Ship Date'] = pd.to_datetime(
    df_clean['Ship Date'],
    format='%d-%m-%Y'
)

# Create Lead Time

In [12]:
df_clean['Lead_Time'] = (
    df_clean['Ship Date']
    -
    df_clean['Order Date']
).dt.days

# create profit margin 

In [13]:
df_clean['Profit_Margin'] = (
    df_clean['Gross Profit']
    /
    df_clean['Sales']
)

# Merge product factory mapping 

In [14]:
df_clean = df_clean.merge(
    factory_map,
    on='Product Name',
    how='left'
)

# Merge factory coordinated 

In [15]:
df_clean = df_clean.merge(
    factory_coords,
    on='Factory',
    how='left'
)

# Outlier detection 

In [16]:
Q1 = df_clean['Lead_Time'].quantile(0.25)
Q3 = df_clean['Lead_Time'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

outliers = df_clean[
    (df_clean['Lead_Time'] < lower)
    |
    (df_clean['Lead_Time'] > upper)
]

print(outliers.shape)

(0, 23)


# Data quality check 

In [18]:
print(df_clean.shape)

print(df_clean.isnull().sum().sum())

print(df_clean.duplicated().sum())

(10194, 23)
0
0


# Save processed dataset 

In [20]:
df_clean.to_csv(
    "../data/processed/cleaned_data.csv",
    index=False
)

In [21]:
saved_df = pd.read_csv(
    "../data/processed/cleaned_data.csv"
)

saved_df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Lead_Time,Profit_Margin,Factory,Latitude,Longitude
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,909,0.649231,Wicked Choccy's,32.076176,-81.088371
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,909,0.653333,Wicked Choccy's,32.076176,-81.088371
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,909,0.713467,Lot's O' Nuts,32.881893,-111.768036
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,909,0.694444,Lot's O' Nuts,32.881893,-111.768036
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,912,0.653333,Wicked Choccy's,32.076176,-81.088371


# End of Notebook

In [22]:
print(
    "Data Cleaning & Feature Creation Completed Successfully"
)

Data Cleaning & Feature Creation Completed Successfully
